# DS200.Q21 — Run Missing Experiments

This notebook runs all missing experiments to fully satisfy course requirements:

| Phase | Purpose | REQ | Time |
|-------|---------|-----|------|
| **1. Error Analysis** | Confusion matrices, per-class F1, bootstrap CI | REQ-03 | ~10 min |
| **2. Ensemble** | Weighted voting (T5 + BERT models) | REQ-04 | ~15 min |
| **3. Focal Loss** | Train T5 with Focal Loss | REQ-04 | ~30 min |
| **4. EDA Augmentation** | Augment minority classes, retrain | REQ-04 | ~30 min |
| **5. Summary & Charts** | Replace projected numbers with real results | — | 2 min |

**Runtime:** GPU (A100 / V100 / T4) on Colab Pro

---
## Phase 0: Setup

In [ ]:
# Clone repo and install dependencies
git push  clone https://github.com/paht2005/DS200.Q21-Group2-Reimplementation-and-Improvement-of-ViHateT5-for-Vietnamese-HSD-Project.git /content/DS200.Q21_Project
%cd /content/DS200.Q21_Project

# Pin transformers <5.0.0 (5.x breaks T5 tokenizer)
pip install -r requirements.txt install -q -r requirements.txt "transformers>=4.40.0,<5.0.0"

In [ ]:
# HuggingFace login (required for gated dataset sonlam1102/vihsd)
import os
from getpass import getpass
from huggingface_hub import login

# Option 1: Colab secrets (if you saved HF_TOKEN in Colab secrets)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Token loaded from Colab secrets")
except Exception:
    # Option 2: Manual input
    hf_token = getpass("Enter your HuggingFace token: ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)
print("HuggingFace login OK")

In [ ]:
# Verify GPU and imports
import torch
import numpy as np
import pandas as pd
import sys
import warnings
warnings.filterwarnings("ignore")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Add src to path
sys.path.insert(0, "src")
os.makedirs("results/analysis", exist_ok=True)
os.makedirs("results/images", exist_ok=True)
print(f"Device: {DEVICE}")
print("Setup complete!")

---
## Phase 1: Error Analysis (REQ-03)

Generate confusion matrices, per-class F1 breakdown, misclassification patterns, and bootstrap 95% CI.

In [ ]:
# Load best T5 model from HuggingFace
from transformers import T5ForConditionalGeneration, AutoTokenizer

MODEL_ID = "NCPhat2005/vit5_finetune_balanced"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f"Model loaded: {MODEL_ID}")

In [ ]:
# Load test datasets
from data_loader import load_vihsd, load_victsd

vihsd_train, vihsd_val, vihsd_test, _ = load_vihsd()
victsd_train, victsd_val, victsd_test, _ = load_victsd()

print(f"ViHSD  test: {len(vihsd_test)} samples")
print(f"ViCTSD test: {len(victsd_test)} samples")

In [ ]:
# T5 batch prediction helper
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

def predict_t5_batch(model, tokenizer, texts, prefix, batch_size=32, device=DEVICE):
    """Generate T5 predictions in batches."""
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc=prefix[:30]):
        batch = [f"{prefix}: {t}" for t in texts[i:i + batch_size]]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=256
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=64, num_beams=1, do_sample=False)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend(decoded)
    return preds

# Label mappings
LABEL_MAP_VIHSD = {"CLEAN": 0, "clean": 0, "OFFENSIVE": 1, "offensive": 1, "HATE": 2, "hate": 2}
LABEL_MAP_VICTSD = {"NONE": 0, "none": 0, "TOXIC": 1, "toxic": 1}

print("Helper defined.")

In [ ]:
# Generate predictions on test sets
vihsd_texts = vihsd_test["free_text"].tolist()
victsd_texts = victsd_test["Comment"].tolist()

print("Generating ViHSD predictions...")
vihsd_raw_preds = predict_t5_batch(model, tokenizer, vihsd_texts, "hate-speech-detection")

print("\nGenerating ViCTSD predictions...")
victsd_raw_preds = predict_t5_batch(model, tokenizer, victsd_texts, "toxic-speech-detection")

# Map to numeric labels
vihsd_preds = [LABEL_MAP_VIHSD.get(p.strip(), 2) for p in vihsd_raw_preds]
vihsd_true = vihsd_test["label_id"].tolist()

victsd_preds = [LABEL_MAP_VICTSD.get(p.strip(), 0) for p in victsd_raw_preds]
victsd_true = victsd_test["Toxicity"].tolist()

# Sanity check
baseline_vihsd_f1 = f1_score(vihsd_true, vihsd_preds, average="macro")
baseline_victsd_f1 = f1_score(victsd_true, victsd_preds, average="macro")
print(f"\nViHSD  Macro F1: {baseline_vihsd_f1:.4f}")
print(f"ViCTSD Macro F1: {baseline_victsd_f1:.4f}")

In [ ]:
# Save predictions for reuse
pd.DataFrame({
    "text": vihsd_texts, "true_label": vihsd_true,
    "pred_label": vihsd_preds, "raw_pred": vihsd_raw_preds,
}).to_csv("results/analysis/vihsd_predictions.csv", index=False)

pd.DataFrame({
    "text": victsd_texts, "true_label": victsd_true,
    "pred_label": victsd_preds, "raw_pred": victsd_raw_preds,
}).to_csv("results/analysis/victsd_predictions.csv", index=False)

print("Saved predictions to results/analysis/")

In [ ]:
# Run full error analysis
import matplotlib
matplotlib.use("Agg")
from error_analysis import run_full_error_analysis

run_full_error_analysis(
    vihsd_true=vihsd_true,
    vihsd_pred=vihsd_preds,
    vihsd_texts=vihsd_texts,
    victsd_true=victsd_true,
    victsd_pred=victsd_preds,
    victsd_texts=victsd_texts,
)
print("\n Phase 1 COMPLETE — Error Analysis")

In [ ]:
# Display generated charts
from IPython.display import Image, display
import glob

for img_path in sorted(glob.glob("results/images/confusion_matrix_*.png") +
                       glob.glob("results/images/per_class_*.png") +
                       glob.glob("results/images/error_distribution*.png")):
    print(f"\n--- {os.path.basename(img_path)} ---")
    display(Image(filename=img_path, width=700))

# Show statistical significance
sig_df = pd.read_csv("results/analysis/statistical_significance.csv")
display(sig_df)

---
## Phase 2: Ensemble Evaluation (REQ-04)

Combine models via optimized weighted voting:
- **ViHSD (3-class):** T5-balanced + T5-hate-only (ViSoBERT is binary, cannot be used here)
- **ViCTSD (binary):** ViSoBERT + T5-balanced

In [ ]:
# Load ViSoBERT labeling model (binary: NONE=0, HATE/TOXIC=1)
from transformers import AutoModelForSequenceClassification

BERT_MODEL = "NCPhat2005/visobert_labeling"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL).to(DEVICE).eval()
print(f"ViSoBERT loaded: {BERT_MODEL} (num_labels={bert_model.config.num_labels})")

In [ ]:
# BERT batch prediction helper
def predict_bert_batch(model, tokenizer, texts, batch_size=32, device=DEVICE):
    all_preds = []
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT inference"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=256
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_probs)

In [ ]:
# --- ViCTSD Ensemble: ViSoBERT (binary) + ViHateT5 (binary) ---
from ensemble import optimize_weights, evaluate_ensemble

print("=" * 60)
print("ViCTSD ENSEMBLE (ViSoBERT + ViHateT5)")
print("=" * 60)

# Get ViSoBERT predictions on ViCTSD test
print("\nViSoBERT on ViCTSD test...")
bert_victsd_preds, _ = predict_bert_batch(bert_model, bert_tokenizer, victsd_texts)
print(f"ViSoBERT ViCTSD Macro F1: {f1_score(victsd_true, bert_victsd_preds, average='macro'):.4f}")

# Optimize weights on validation set
print("\nOptimizing weights on validation set...")
victsd_val_texts = victsd_val["Comment"].tolist()
victsd_val_true = np.array(victsd_val["Toxicity"].tolist())

t5_val_raw = predict_t5_batch(model, tokenizer, victsd_val_texts, "toxic-speech-detection")
t5_val_preds = np.array([LABEL_MAP_VICTSD.get(p.strip(), 0) for p in t5_val_raw])
bert_val_preds, _ = predict_bert_batch(bert_model, bert_tokenizer, victsd_val_texts)

victsd_val_predictions = {"ViHateT5": t5_val_preds, "ViSoBERT": bert_val_preds}
victsd_best_weights = optimize_weights(victsd_val_predictions, victsd_val_true, num_classes=2, n_trials=2000)

In [ ]:
# Apply optimal weights on ViCTSD test set
t5_victsd_preds = np.array(victsd_preds)
victsd_individual = {"ViHateT5": t5_victsd_preds, "ViSoBERT": bert_victsd_preds}

# Weighted voting
n_samples = len(victsd_true)
weighted_votes = np.zeros((n_samples, 2))
for name, preds in victsd_individual.items():
    w = victsd_best_weights[name]
    for i, pred in enumerate(preds):
        if 0 <= pred < 2:
            weighted_votes[i, pred] += w
victsd_ensemble_preds = np.argmax(weighted_votes, axis=1)

evaluate_ensemble(victsd_ensemble_preds, np.array(victsd_true), victsd_individual, "ViCTSD — Weighted Voting")
ensemble_victsd_f1 = f1_score(victsd_true, victsd_ensemble_preds, average="macro")
print(f"\nEnsemble ViCTSD Macro F1: {ensemble_victsd_f1:.4f}")

In [ ]:
# Free BERT model memory
del bert_model, bert_tokenizer
torch.cuda.empty_cache()

# --- ViHSD Ensemble: T5-balanced + T5-hate-only (both 3-class) ---
print("\n" + "=" * 60)
print("ViHSD ENSEMBLE (T5-balanced + T5-hate-only)")
print("=" * 60)

MODEL_HATE = "NCPhat2005/vit5_finetune_hate_only"
model_hate = T5ForConditionalGeneration.from_pretrained(MODEL_HATE).to(DEVICE).eval()
tok_hate = AutoTokenizer.from_pretrained(MODEL_HATE)

print("\nT5-hate-only on ViHSD test...")
hate_raw_preds = predict_t5_batch(model_hate, tok_hate, vihsd_texts, "hate-speech-detection")
hate_preds = np.array([LABEL_MAP_VIHSD.get(p.strip(), 2) for p in hate_raw_preds])
print(f"T5-hate-only ViHSD Macro F1: {f1_score(vihsd_true, hate_preds, average='macro'):.4f}")

# Optimize on validation set
print("\nOptimizing weights on validation set...")
vihsd_val_texts = vihsd_val["free_text"].tolist()
vihsd_val_true = np.array(vihsd_val["label_id"].tolist())

bal_val_raw = predict_t5_batch(model, tokenizer, vihsd_val_texts, "hate-speech-detection")
bal_val_preds = np.array([LABEL_MAP_VIHSD.get(p.strip(), 2) for p in bal_val_raw])
hate_val_raw = predict_t5_batch(model_hate, tok_hate, vihsd_val_texts, "hate-speech-detection")
hate_val_preds = np.array([LABEL_MAP_VIHSD.get(p.strip(), 2) for p in hate_val_raw])

vihsd_val_predictions = {"T5-balanced": bal_val_preds, "T5-hate-only": hate_val_preds}
vihsd_best_weights = optimize_weights(vihsd_val_predictions, vihsd_val_true, num_classes=3, n_trials=2000)

In [ ]:
# Apply to ViHSD test set
t5_balanced_preds = np.array(vihsd_preds)
vihsd_individual = {"T5-balanced": t5_balanced_preds, "T5-hate-only": hate_preds}

n = len(vihsd_true)
wv = np.zeros((n, 3))
for name, preds in vihsd_individual.items():
    w = vihsd_best_weights[name]
    for i, pred in enumerate(preds):
        if 0 <= pred < 3:
            wv[i, pred] += w
vihsd_ensemble_preds = np.argmax(wv, axis=1)

evaluate_ensemble(vihsd_ensemble_preds, np.array(vihsd_true), vihsd_individual, "ViHSD — Weighted Voting")
ensemble_vihsd_f1 = f1_score(vihsd_true, vihsd_ensemble_preds, average="macro")
print(f"\nEnsemble ViHSD Macro F1: {ensemble_vihsd_f1:.4f}")

# Free hate-only model
del model_hate, tok_hate
torch.cuda.empty_cache()
print("\n Phase 2 COMPLETE — Ensemble Evaluation")

---
## Phase 3: Focal Loss Training (REQ-04)

Train T5 with Focal Loss: $FL(p_t) = -(1-p_t)^\gamma \log(p_t)$ with $\gamma=2.0$ and label smoothing $\epsilon=0.1$.

In [ ]:
# Prepare multi-task source/target data
from datasets import Dataset

def prepare_vihsd(df):
    label_map = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}
    out = df.copy()
    out["source"] = out["free_text"].apply(lambda x: f"hate-speech-detection: {x}")
    out["target"] = out["label_id"].apply(lambda x: label_map[x])
    return out[["source", "target"]]

def prepare_victsd(df):
    label_map = {0: "NONE", 1: "TOXIC"}
    out = df.copy()
    out["source"] = out["Comment"].apply(lambda x: f"toxic-speech-detection: {x}")
    out["target"] = out["Toxicity"].apply(lambda x: label_map[x])
    return out[["source", "target"]]

train_df = pd.concat([prepare_vihsd(vihsd_train), prepare_victsd(victsd_train)], ignore_index=True)
val_df = pd.concat([prepare_vihsd(vihsd_val), prepare_victsd(victsd_val)], ignore_index=True)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Label distribution:\n{train_df['target'].value_counts()}")

In [ ]:
# Tokenize for Seq2Seq training
CKPT = "NCPhat2005/vit5_finetune_balanced"  # Start from best checkpoint
focal_tokenizer = AutoTokenizer.from_pretrained(CKPT)
focal_model = T5ForConditionalGeneration.from_pretrained(CKPT).to(DEVICE)

def preprocess(examples):
    inputs = focal_tokenizer(examples["source"], max_length=256, truncation=True, padding="max_length")
    targets = focal_tokenizer(examples["target"], max_length=64, truncation=True, padding="max_length")
    labels = targets["input_ids"]
    labels = [[(l if l != focal_tokenizer.pad_token_id else -100) for l in label] for label in labels]
    inputs["labels"] = labels
    return inputs

train_tok = Dataset.from_pandas(train_df).map(preprocess, batched=True, remove_columns=["source", "target"])
val_tok = Dataset.from_pandas(val_df).map(preprocess, batched=True, remove_columns=["source", "target"])
print(f"Tokenized: train={len(train_tok)}, val={len(val_tok)}")

In [ ]:
# Train with Focal Loss
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from focal_loss import FocalLossSeq2SeqTrainer

focal_args = Seq2SeqTrainingArguments(
    output_dir="models/vit5_focal_loss",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_strategy="epoch",
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    predict_with_generate=True,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available(),
)

focal_trainer = FocalLossSeq2SeqTrainer(
    model=focal_model,
    args=focal_args,
    data_collator=DataCollatorForSeq2Seq(focal_tokenizer, model=focal_model),
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=focal_tokenizer,
    focal_gamma=2.0,
    label_smoothing=0.1,
)

print("Training with Focal Loss (gamma=2.0, smoothing=0.1)...")
focal_trainer.train()
focal_trainer.save_model("models/vit5_focal_loss")
focal_tokenizer.save_pretrained("models/vit5_focal_loss")
print("\nFocal Loss training complete!")

In [ ]:
# Evaluate Focal Loss model
focal_eval_model = T5ForConditionalGeneration.from_pretrained("models/vit5_focal_loss").to(DEVICE).eval()
focal_eval_tok = AutoTokenizer.from_pretrained("models/vit5_focal_loss")

focal_vihsd_raw = predict_t5_batch(focal_eval_model, focal_eval_tok, vihsd_texts, "hate-speech-detection")
focal_vihsd_preds = [LABEL_MAP_VIHSD.get(p.strip(), 2) for p in focal_vihsd_raw]
focal_vihsd_f1 = f1_score(vihsd_true, focal_vihsd_preds, average="macro")

focal_victsd_raw = predict_t5_batch(focal_eval_model, focal_eval_tok, victsd_texts, "toxic-speech-detection")
focal_victsd_preds = [LABEL_MAP_VICTSD.get(p.strip(), 0) for p in focal_victsd_raw]
focal_victsd_f1 = f1_score(victsd_true, focal_victsd_preds, average="macro")

print("\n" + "=" * 60)
print("FOCAL LOSS RESULTS")
print("=" * 60)
print(f"ViHSD  Macro F1: {focal_vihsd_f1:.4f}  (baseline: {baseline_vihsd_f1:.4f})")
print(f"ViCTSD Macro F1: {focal_victsd_f1:.4f}  (baseline: {baseline_victsd_f1:.4f})")
print("=" * 60)

del focal_model, focal_eval_model
torch.cuda.empty_cache()
print("\n Phase 3 COMPLETE — Focal Loss")

---
## Phase 4: EDA Augmentation Training (REQ-04)

Easy Data Augmentation: synonym replacement, random insertion/swap/deletion on minority classes, then retrain T5.

In [ ]:
# Augment training data
from augment import augment_vihsd, augment_victsd
import random
random.seed(42)

print("Original ViHSD distribution:")
print(vihsd_train["label_id"].value_counts())

print("\nAugmenting ViHSD (target_ratio=0.8)...")
vihsd_aug = augment_vihsd(vihsd_train.copy(), target_ratio=0.8)

print("\nAugmenting ViCTSD (target_ratio=0.8)...")
victsd_aug = augment_victsd(victsd_train.copy(), target_ratio=0.8)

print(f"\nViHSD:  {len(vihsd_train)} -> {len(vihsd_aug)} (+{len(vihsd_aug) - len(vihsd_train)})")
print(f"ViCTSD: {len(victsd_train)} -> {len(victsd_aug)} (+{len(victsd_aug) - len(victsd_train)})")

In [ ]:
# Prepare and tokenize augmented data
train_aug_df = pd.concat([prepare_vihsd(vihsd_aug), prepare_victsd(victsd_aug)], ignore_index=True)
print(f"Augmented training samples: {len(train_aug_df)}")
print(f"Label distribution:\n{train_aug_df['target'].value_counts()}")

train_aug_tok = Dataset.from_pandas(train_aug_df).map(preprocess, batched=True, remove_columns=["source", "target"])
print(f"Tokenized: {len(train_aug_tok)}")

In [ ]:
# Train on augmented data (standard CrossEntropy)
from transformers import Seq2SeqTrainer

eda_model = T5ForConditionalGeneration.from_pretrained(CKPT).to(DEVICE)
eda_tokenizer = AutoTokenizer.from_pretrained(CKPT)

eda_args = Seq2SeqTrainingArguments(
    output_dir="models/vit5_eda_augment",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_strategy="epoch",
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    predict_with_generate=True,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available(),
)

eda_trainer = Seq2SeqTrainer(
    model=eda_model,
    args=eda_args,
    data_collator=DataCollatorForSeq2Seq(eda_tokenizer, model=eda_model),
    train_dataset=train_aug_tok,
    eval_dataset=val_tok,
    processing_class=eda_tokenizer,
)

print("Training on EDA-augmented data...")
eda_trainer.train()
eda_trainer.save_model("models/vit5_eda_augment")
eda_tokenizer.save_pretrained("models/vit5_eda_augment")
print("\nEDA training complete!")

In [ ]:
# Evaluate EDA model
eda_eval_model = T5ForConditionalGeneration.from_pretrained("models/vit5_eda_augment").to(DEVICE).eval()
eda_eval_tok = AutoTokenizer.from_pretrained("models/vit5_eda_augment")

eda_vihsd_raw = predict_t5_batch(eda_eval_model, eda_eval_tok, vihsd_texts, "hate-speech-detection")
eda_vihsd_preds = [LABEL_MAP_VIHSD.get(p.strip(), 2) for p in eda_vihsd_raw]
eda_vihsd_f1 = f1_score(vihsd_true, eda_vihsd_preds, average="macro")

eda_victsd_raw = predict_t5_batch(eda_eval_model, eda_eval_tok, victsd_texts, "toxic-speech-detection")
eda_victsd_preds = [LABEL_MAP_VICTSD.get(p.strip(), 0) for p in eda_victsd_raw]
eda_victsd_f1 = f1_score(victsd_true, eda_victsd_preds, average="macro")

print("\n" + "=" * 60)
print("EDA AUGMENTATION RESULTS")
print("=" * 60)
print(f"ViHSD  Macro F1: {eda_vihsd_f1:.4f}  (baseline: {baseline_vihsd_f1:.4f})")
print(f"ViCTSD Macro F1: {eda_victsd_f1:.4f}  (baseline: {baseline_victsd_f1:.4f})")
print("=" * 60)

del eda_model, eda_eval_model
torch.cuda.empty_cache()
print("\n Phase 4 COMPLETE — EDA Augmentation")

---
## Phase 5: Summary & Real Results Chart

In [ ]:
# Full results summary
print("\n" + "=" * 70)
print("FULL RESULTS SUMMARY")
print("=" * 70)
print(f"{'Method':<30} {'ViHSD F1':>10} {'ViCTSD F1':>10}")
print("-" * 50)
print(f"{'Baseline (ViT5-base)':      <30} {'0.6625':>10} {'0.7163':>10}")
print(f"{'ViHateT5 (balanced)':       <30} {baseline_vihsd_f1:>10.4f} {baseline_victsd_f1:>10.4f}")
print(f"{'+ Focal Loss':              <30} {focal_vihsd_f1:>10.4f} {focal_victsd_f1:>10.4f}")
print(f"{'+ EDA Augmentation':        <30} {eda_vihsd_f1:>10.4f} {eda_victsd_f1:>10.4f}")
print(f"{'+ Ensemble':                <30} {ensemble_vihsd_f1:>10.4f} {ensemble_victsd_f1:>10.4f}")
print("=" * 70)

In [ ]:
# Generate improvement comparison chart with REAL numbers
import matplotlib.pyplot as plt

methods = [
    "ViT5-base\n(Baseline)",
    "ViHateT5\n(Reimpl)",
    "+ Focal Loss\n(Proposed)",
    "+ EDA Augment\n(Proposed)",
    "+ Ensemble\n(Proposed)",
]

vihsd_scores  = [0.6625, baseline_vihsd_f1, focal_vihsd_f1, eda_vihsd_f1, ensemble_vihsd_f1]
victsd_scores = [0.7163, baseline_victsd_f1, focal_victsd_f1, eda_victsd_f1, ensemble_victsd_f1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tasks = ["ViHSD (Hate Speech)", "ViCTSD (Toxic Speech)"]
all_scores = [vihsd_scores, victsd_scores]
colors = ["#95a5a6", "#3498db", "#e74c3c", "#9b59b6", "#f39c12"]

for i, (task, scores) in enumerate(zip(tasks, all_scores)):
    ax = axes[i]
    bars = ax.bar(range(len(methods)), scores, color=colors, edgecolor="black", linewidth=0.5)
    for bar, val in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, fontsize=8, ha="center")
    ax.set_ylabel("Macro F1-Score")
    ax.set_title(task, fontsize=12, fontweight="bold")
    ax.set_ylim(min(scores) - 0.03, max(scores) + 0.02)
    ax.axhline(y=scores[0], color="gray", linestyle="--", alpha=0.5)

fig.suptitle("Improvement Results — Proposed Methods vs Baseline (REAL)", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig("results/images/improvement_comparison_real.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/images/improvement_comparison_real.png")

In [ ]:
# Save results CSV
results_df = pd.DataFrame({
    "Method": ["ViT5-base (Baseline)", "ViHateT5 (Balanced Reimpl)",
               "+ Focal Loss (gamma=2.0)", "+ EDA Augmentation", "+ Ensemble (Weighted Voting)"],
    "ViHSD_Macro_F1": [0.6625, baseline_vihsd_f1, focal_vihsd_f1, eda_vihsd_f1, ensemble_vihsd_f1],
    "ViCTSD_Macro_F1": [0.7163, baseline_victsd_f1, focal_victsd_f1, eda_victsd_f1, ensemble_victsd_f1],
})
results_df["Average_F1"] = (results_df["ViHSD_Macro_F1"] + results_df["ViCTSD_Macro_F1"]) / 2
results_df.to_csv("results/analysis/improvement_results.csv", index=False)

print(results_df.to_string(index=False))
print("\nSaved to: results/analysis/improvement_results.csv")

In [ ]:
# Print values to update src/visualize.py
print("\n" + "=" * 70)
print("COPY THESE VALUES to src/visualize.py (lines ~200-202):")
print("=" * 70)
print(f'vihsd_scores  = [0.6625, {baseline_vihsd_f1:.4f}, {baseline_vihsd_f1:.4f}, {focal_vihsd_f1:.4f}, {eda_vihsd_f1:.4f}, {ensemble_vihsd_f1:.4f}]')
print(f'victsd_scores = [0.7163, {baseline_victsd_f1:.4f}, {baseline_victsd_f1:.4f}, {focal_victsd_f1:.4f}, {eda_victsd_f1:.4f}, {ensemble_victsd_f1:.4f}]')
print("=" * 70)

In [ ]:
# List all generated output files
print("\n" + "=" * 70)
print("ALL GENERATED OUTPUT FILES")
print("=" * 70)
for pattern in ["results/analysis/*", "results/images/confusion_*",
                "results/images/per_class_*", "results/images/error_*",
                "results/images/improvement_comparison_real*"]:
    for f in sorted(glob.glob(pattern)):
        size = os.path.getsize(f)
        print(f"  {f} ({size:,} bytes)")

print("\n" + "=" * 70)
print("ALL PHASES COMPLETE!")
print("=" * 70)
print("\nNext steps:")
print("1. Download results/ folder from Colab")
print("2. Copy into local project")
print("3. Update src/visualize.py with real numbers (printed above)")
print("4. Update LaTeX report (docs/DS200.Q21-Group2.tex) with results tables")

In [ ]:
# Zip results for easy download
!zip -r /content/results_output.zip results/analysis/ results/images/confusion_matrix_*.png results/images/per_class_*.png results/images/error_*.png results/images/improvement_comparison_real.png
print("\nDownload: /content/results_output.zip")

from google.colab import files
files.download("/content/results_output.zip")

In [ ]:
# Optional: Push new models to HuggingFace
# Uncomment to push focal loss and EDA models

# from huggingface_hub import HfApi
# api = HfApi()
#
# api.upload_folder(
#     folder_path="models/vit5_focal_loss",
#     repo_id="NCPhat2005/vit5_focal_loss",
#     repo_type="model",
#     create_pr=False,
# )
#
# api.upload_folder(
#     folder_path="models/vit5_eda_augment",
#     repo_id="NCPhat2005/vit5_eda_augment",
#     repo_type="model",
#     create_pr=False,
# )
#
# print("Models pushed to HuggingFace!")